In [1]:

import os
import cv2
import numpy as np
import torch
import glob
from tqdm import tqdm
import traceback
import matplotlib.pyplot as plt


# Import Segment Anything
from segment_anything import sam_model_registry, SamPredictor

# Setup SAM model
def setup_sam(checkpoint_path="sam_vit_h_4b8939.pth", model_type="vit_h", device="cuda"):
    print(f"Loading SAM model ({model_type})...")
    try:
        sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
        sam.to(device=device)
        predictor = SamPredictor(sam)
        print("SAM model loaded successfully!")
        return predictor
    except Exception as e:
        print(f"Error loading SAM model: {str(e)}")
        traceback.print_exc()
        return None

# Convert YOLO bounding box format to pixel coordinates
def yolo_to_pixel_bbox(yolo_bbox, img_width, img_height):
    """
    Convert YOLO format (x_center, y_center, width, height) to
    pixel coordinates (x_min, y_min, x_max, y_max)
    """
    x_center, y_center, width, height = yolo_bbox
    x_center *= img_width
    y_center *= img_height
    width *= img_width
    height *= img_height

    x_min = max(0, int(x_center - width / 2))
    y_min = max(0, int(y_center - height / 2))
    x_max = min(img_width, int(x_center + width / 2))
    y_max = min(img_height, int(y_center + height / 2))

    return [x_min, y_min, x_max, y_max]

# Convert segmentation mask to YOLO polygon format
def mask_to_yolo_polygon(mask, img_width, img_height, simplify=True, tolerance=0.01):
    """
    Convert binary mask to YOLO polygon format (normalized coordinates)
    If simplify is True, the polygon will be simplified using Douglas-Peucker algorithm
    """
    try:
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            print("  No contours found in mask")
            return None

        # Get the largest contour
        largest_contour = max(contours, key=cv2.contourArea)

        if len(largest_contour) < 3:
            print("  Contour has less than 3 points")
            return None

        if simplify:
            # Simplify the polygon to reduce the number of points
            epsilon = tolerance * cv2.arcLength(largest_contour, True)
            approx = cv2.approxPolyDP(largest_contour, epsilon, True)
            polygon = approx.flatten().reshape(-1, 2)
        else:
            polygon = largest_contour.flatten().reshape(-1, 2)

        # Normalize coordinates
        normalized_polygon = []
        for x, y in polygon:
            normalized_polygon.extend([float(x) / img_width, float(y) / img_height])

        return normalized_polygon
    except Exception as e:
        print(f"  Error in mask_to_yolo_polygon: {str(e)}")
        traceback.print_exc()
        return None

# Process a single image with its bounding boxes
def process_image(image_path, label_path, predictor, output_label_path=None, save_debug_viz=False):
    """
    Process an image and its bounding boxes to generate segmentation masks
    Returns output label path if successful
    """
    print(f"Processing {os.path.basename(image_path)}...")

    try:
        # Read image
        image = cv2.imread(image_path)
        if image is None:
            print(f"  Error: Could not read image {image_path}")
            return None

        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]
        print(f"  Image size: {width}x{height}")

        # Set image in predictor
        try:
            predictor.set_image(image_rgb)
        except Exception as e:
            print(f"  Error setting image in predictor: {str(e)}")
            traceback.print_exc()
            return None

        # Read YOLO format labels
        if not os.path.exists(label_path):
            print(f"  Error: Label file not found: {label_path}")
            return None

        with open(label_path, 'r') as f:
            lines = f.readlines()

        print(f"  Found {len(lines)} objects in label file")

        new_lines = []
        success_count = 0
        fallback_count = 0

        # Create debug image if needed
        if save_debug_viz:
            debug_img = image.copy()

        for i, line in enumerate(lines):
            try:
                parts = line.strip().split()
                if len(parts) < 5:
                    print(f"  Warning: Line {i+1} has fewer than 5 parts, skipping")
                    continue

                class_id = parts[0]
                bbox = [float(coord) for coord in parts[1:5]]
                pixel_bbox = yolo_to_pixel_bbox(bbox, width, height)

                # Add bbox to debug image
                if save_debug_viz:
                    x_min, y_min, x_max, y_max = pixel_bbox
                    cv2.rectangle(debug_img, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)

                # Convert to input format expected by SAM
                box = np.array(pixel_bbox)

                # Check if box is valid
                if box[0] >= box[2] or box[1] >= box[3]:
                    print(f"  Warning: Invalid box for object {i+1}: {box}")
                    new_lines.append(line.strip())
                    fallback_count += 1
                    continue

                # Generate masks
                print(f"  Generating mask for object {i+1}...")
                try:
                    masks, scores, _ = predictor.predict(
                        box=box,
                        multimask_output=True,  # Get multiple masks
                        point_coords=None,
                        point_labels=None
                    )

                    # Get the best mask
                    best_mask_idx = np.argmax(scores)
                    best_mask = masks[best_mask_idx]

                    # Debug info
                    mask_area = best_mask.sum()
                    print(f"  Mask area: {mask_area} pixels")

                    if mask_area < 10:
                        print(f"  Warning: Mask too small for object {i+1}")
                        new_lines.append(line.strip())
                        fallback_count += 1
                        continue

                    # Convert mask to YOLO polygon format
                    polygon = mask_to_yolo_polygon(best_mask, width, height)

                    if polygon is None:
                        print(f"  Warning: Failed to convert mask to polygon for object {i+1}")
                        new_lines.append(line.strip())
                        fallback_count += 1
                    elif len(polygon) < 6:  # At least 3 points needed
                        print(f"  Warning: Polygon has only {len(polygon)//2} points for object {i+1}")
                        new_lines.append(line.strip())
                        fallback_count += 1
                    else:
                        # Format: class_id x1 y1 x2 y2 ... xn yn
                        new_line = f"{class_id} " + " ".join([f"{coord:.6f}" for coord in polygon])
                        new_lines.append(new_line)
                        success_count += 1

                        # Add polygon to debug image
                        if save_debug_viz:
                            poly_points = []
                            for j in range(0, len(polygon), 2):
                                x = int(polygon[j] * width)
                                y = int(polygon[j+1] * height)
                                poly_points.append([x, y])

                            cv2.polylines(debug_img, [np.array(poly_points)], True, (0, 0, 255), 2)

                except Exception as e:
                    print(f"  Error in SAM prediction for object {i+1}: {str(e)}")
                    traceback.print_exc()
                    new_lines.append(line.strip())
                    fallback_count += 1

            except Exception as e:
                print(f"  Error processing object {i+1}: {str(e)}")
                traceback.print_exc()
                # Add original line if we can
                if line.strip():
                    new_lines.append(line.strip())
                    fallback_count += 1

        # Write the new labels
        if output_label_path is None:
            output_label_path = label_path.replace('.txt', '_seg.txt')

        with open(output_label_path, 'w') as f:
            for line in new_lines:
                f.write(line + '\n')

        print(f"  Results: {success_count} segmentations, {fallback_count} fallbacks to bboxes")

        # Save debug visualization
        if save_debug_viz and (success_count > 0 or fallback_count > 0):
            os.makedirs("readme_images", exist_ok=True)
            base_name = os.path.splitext(os.path.basename(image_path))[0]
            debug_path = os.path.join("readme_images", base_name + "_debug.jpg")
            cv2.imwrite(debug_path, debug_img)
            print(f"  Debug visualization saved to {debug_path}")

        return output_label_path if (success_count > 0 or fallback_count > 0) else None

    except Exception as e:
        print(f"Error processing image {image_path}: {str(e)}")
        traceback.print_exc()
        return None

def convert_dataset(
    images_dir,
    labels_dir,
    output_labels_dir=None,
    sam_checkpoint="sam_vit_h_4b8939.pth",
    model_type="vit_h",
    device="cuda",
    debug_first_n=3  # Save debug visualizations for first N images
):
    """
    Convert an entire YOLO dataset from bounding boxes to segmentation
    """
    # Setup output directory
    if output_labels_dir is None:
        output_labels_dir = labels_dir + "_segmentation"

    os.makedirs(output_labels_dir, exist_ok=True)

    # Setup SAM
    predictor = setup_sam(sam_checkpoint, model_type, device)
    if predictor is None:
        print("Failed to load SAM model. Exiting.")
        return None

    # Get all image files
    image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
    image_paths = []
    for ext in image_extensions:
        image_paths.extend(glob.glob(os.path.join(images_dir, ext)))

    image_paths = image_paths[:5]

    print(f"Found {len(image_paths)} images")

    # Process each image
    processed = 0
    successful = 0

    for i, image_path in enumerate(tqdm(image_paths)):
        save_debug = (i < debug_first_n)  # Save debug visualization for first N images

        image_filename = os.path.basename(image_path)
        base_filename = os.path.splitext(image_filename)[0]
        label_path = os.path.join(labels_dir, base_filename + '.txt')
        output_path = os.path.join(output_labels_dir, base_filename + '.txt')

        if os.path.exists(label_path):
            processed += 1
            output_label = process_image(
                image_path,
                label_path,
                predictor,
                output_path,
                save_debug_viz=save_debug
            )
            if output_label:
                successful += 1

    print(f"Conversion complete. Segmentation labels saved to {output_labels_dir}")
    print(f"Successfully processed {successful}/{processed} images with labels")

    return output_labels_dir

# Main function for Colab
def main(images_dir, labels_dir, output_dir, model_type="vit_b"):
    print(f"\n=== YOLOv8 Bounding Box to Segmentation Converter ===\n")

    # Check if SAM model exists, if not download it
    if model_type == "vit_h":
        model_path = "sam_vit_h_4b8939.pth"
        download_url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
    else:
        model_path = "sam_vit_b_01ec64.pth"
        download_url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"

    if not os.path.exists(model_path):
        print(f"Downloading SAM model ({model_type})...")
        !wget {download_url}

    # Run conversion with more detailed logging
    print(f"Starting conversion with {model_type} model...")

    # Use CPU if CUDA not available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    output_dir = convert_dataset(
        images_dir=images_dir,
        labels_dir=labels_dir,
        output_labels_dir=output_dir,
        sam_checkpoint=model_path,
        model_type=model_type,
        device=device,
        debug_first_n=3  # Save debug visualizations for first 3 images
    )

    if output_dir:
        print(f"\nConversion complete! Segmentation labels saved to: {output_dir}")
        print("\n=== How to use the converted dataset with YOLOv8 ===")
        print("1. Create a new dataset.yaml file with the segmentation labels directory")
        print("2. Train a YOLOv8 segmentation model with:")
        print("   !yolo segment train data=dataset.yaml model=yolov8s-seg.pt epochs=50")
    else:
        print("\nConversion failed. Please check the error messages above.")


if __name__ == "__main__":
    IMAGES_DIR = "dataset/images"
    LABELS_DIR = "dataset/labels"
    OUTPUT_DIR = "seg_dataset/labels"  # labels go here

    MODEL_TYPE = "vit_b"

    main(IMAGES_DIR, LABELS_DIR, OUTPUT_DIR, MODEL_TYPE)

    # Copy images to seg_dataset/images
    import shutil
    os.makedirs("seg_dataset/images", exist_ok=True)
    for img_path in glob.glob("dataset/images/*"):
        shutil.copy(img_path, "seg_dataset/images/")

    print("Images copied to seg_dataset/images")


=== YOLOv8 Bounding Box to Segmentation Converter ===

Starting conversion with vit_b model...
Using device: cpu
Loading SAM model (vit_b)...
SAM model loaded successfully!
Found 5 images


  0%|          | 0/5 [00:00<?, ?it/s]

Processing img_329.jpg...
  Image size: 1280x720


 20%|██        | 1/5 [00:05<00:20,  5.19s/it]

  Found 2 objects in label file
  Generating mask for object 1...
  Mask area: 1036 pixels
  Generating mask for object 2...
  Mask area: 586 pixels
  Results: 2 segmentations, 0 fallbacks to bboxes
  Debug visualization saved to readme_images/img_329_debug.jpg
Processing img_473.jpg...
  Image size: 1280x720


...

 80%|████████  | 4/5 [00:17<00:04,  4.12s/it]

  Found 2 objects in label file
  Generating mask for object 1...
  Mask area: 1912 pixels
  Generating mask for object 2...
  Mask area: 2689 pixels
  Results: 2 segmentations, 0 fallbacks to bboxes
Processing img_467.jpg...
  Image size: 1280x720


100%|██████████| 5/5 [00:21<00:00,  4.23s/it]

  Found 2 objects in label file
  Generating mask for object 1...
  Mask area: 533 pixels
  Generating mask for object 2...
  Mask area: 345 pixels
  Results: 2 segmentations, 0 fallbacks to bboxes
Conversion complete. Segmentation labels saved to seg_dataset/labels
Successfully processed 5/5 images with labels

Conversion complete! Segmentation labels saved to: seg_dataset/labels

=== How to use the converted dataset with YOLOv8 ===
1. Create a new dataset.yaml file with the segmentation labels directory
2. Train a YOLOv8 segmentation model with:
   !yolo segment train data=dataset.yaml model=yolov8s-seg.pt epochs=50


Images copied to seg_dataset/images
